# Level 1: AdamW versus AdamW + WWPGD

Matched-seed validation curves, endpoint statistics, and layerwise WeightWatcher alpha trajectories. Shaded regions are mean ± one sample standard deviation across seeds.

In [ ]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

baseline_root = Path(os.environ['NANOGPT_LEVEL1_BASELINE_RESULTS_ROOT'])
wwpgd_root = Path(os.environ['NANOGPT_LEVEL1_WWPGD_RESULTS_ROOT'])
seed_re = re.compile(r'seed_(\d+)$')

def runs(root):
    out = {}
    for p in sorted(root.iterdir()):
        m = seed_re.search(p.name)
        if m and (p / 'metrics.csv').exists(): out[int(m.group(1))] = p
    return out

B, W = runs(baseline_root), runs(wwpgd_root)
seeds = sorted(set(B) & set(W))
assert seeds, 'No completed matched seeds found'
print('Matched seeds:', seeds)

In [ ]:
def metric_frame(path):
    df = pd.read_csv(path / 'metrics.csv')
    step = next(c for c in ('step','optimizer_step') if c in df.columns)
    loss = next(c for c in ('val_loss','validation_loss') if c in df.columns)
    return df[[step, loss]].rename(columns={step:'step', loss:'val_loss'})

def aggregate(paths):
    frames=[]
    for seed,p in paths.items():
        f=metric_frame(p); f['seed']=seed; frames.append(f)
    long=pd.concat(frames, ignore_index=True)
    return long, long.groupby('step').val_loss.agg(['mean','std','count']).reset_index()

blong, bagg = aggregate({s:B[s] for s in seeds})
wlong, wagg = aggregate({s:W[s] for s in seeds})
fig, ax = plt.subplots(figsize=(12,6))
for label,a in [('adamw',bagg),('adamw_wwpgd',wagg)]:
    ax.plot(a.step,a['mean'],label=label)
    ax.fill_between(a.step,a['mean']-a['std'].fillna(0),a['mean']+a['std'].fillna(0),alpha=.18)
ax.set(title='Level 1 validation loss: mean ± 1 std',xlabel='optimizer step',ylabel='validation loss')
ax.legend(); ax.grid(alpha=.25); plt.show()

In [ ]:
rows=[]
for s in seeds:
    b=metric_frame(B[s]).iloc[-1].val_loss
    w=metric_frame(W[s]).iloc[-1].val_loss
    rows.append({'seed':s,'adamw_final':b,'wwpgd_final':w,'delta_wwpgd_minus_adamw':w-b})
paired=pd.DataFrame(rows)
display(paired)
d=paired.delta_wwpgd_minus_adamw
sem=d.std(ddof=1)/np.sqrt(len(d)) if len(d)>1 else np.nan
print(f'Mean paired delta: {d.mean():.4f}')
print(f'SD paired delta:   {d.std(ddof=1):.4f}')
print(f'Approx 95% CI:     [{d.mean()-1.96*sem:.4f}, {d.mean()+1.96*sem:.4f}]')

In [ ]:
def alpha_long(run_map, arm):
    rows=[]
    for seed,p in run_map.items():
        for f in sorted(p.glob('weightwatcher_step_*.csv')):
            step=int(re.search(r'(\d+)',f.stem).group(1))
            df=pd.read_csv(f)
            alpha_col=next((c for c in ('alpha','alpha_weighted') if c in df.columns),None)
            layer_col=next((c for c in ('name','layer_name','layer_id','layer') if c in df.columns),None)
            if alpha_col and layer_col:
                for _,r in df[[layer_col,alpha_col]].dropna().iterrows():
                    rows.append({'arm':arm,'seed':seed,'step':step,'layer':str(r[layer_col]),'alpha':float(r[alpha_col])})
    return pd.DataFrame(rows)

alphas=pd.concat([alpha_long({s:B[s] for s in seeds},'adamw'),alpha_long({s:W[s] for s in seeds},'adamw_wwpgd')],ignore_index=True)
if alphas.empty:
    print('No WeightWatcher CSVs found yet.')
else:
    summary=alphas.groupby(['arm','layer','step']).alpha.agg(['mean','std']).reset_index()
    for arm in summary.arm.unique():
        fig,ax=plt.subplots(figsize=(14,7))
        sub=summary[summary.arm==arm]
        for layer,g in sub.groupby('layer'):
            g=g.sort_values('step'); ax.plot(g.step,g['mean'],label=layer,linewidth=1)
            ax.fill_between(g.step,g['mean']-g['std'].fillna(0),g['mean']+g['std'].fillna(0),alpha=.08)
        ax.axhline(2.0,linestyle='--',linewidth=1,label='target alpha=2')
        ax.set(title=f'{arm}: all matrix alphas, mean ± 1 std',xlabel='optimizer step',ylabel='WeightWatcher alpha')
        ax.legend(fontsize=7,ncol=2,bbox_to_anchor=(1.02,1),loc='upper left'); ax.grid(alpha=.2); plt.tight_layout(); plt.show()